# 00 · End-to-end: game video → tracking data, touch-by-touch events, 3D play

Runs the whole CourtVision pipeline on one video in Colab:

1. detect + track players, referees, ball and hoop
2. court keypoints → homography → camera (per frame)
3. clean the ball track (select, de-outlier, interpolate)
4. assign teams by jersey colour
5. ball possession with hysteresis
6. passes, interceptions, turnovers, shots (made/missed, 2PT/3PT), rebounds
7. court positions in metres, speed and distance
8. 3D ball height from physics, player body poses (optional), animated 3D scene
9. **touch-by-touch table**: one row per touch with the player, the action, and every other player's position and speed

The touch-by-touch table (`touches.parquet` / `touches.csv`) is the main output. Weights come from notebooks 01 and 02, saved in `MyDrive/courtvision/models/`.

In [ ]:
# Runtime > Change runtime type > GPU before running.
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"
import importlib, sys
if os.path.isdir("/content/courtvision/.git"):
    # Re-run: make the Colab copy match GitHub exactly (edits made inside it are discarded).
    !cd /content/courtvision && git fetch -q origin {BRANCH} && git checkout -q -f -B {BRANCH} FETCH_HEAD
else:
    !git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
%cd /content/courtvision
!pip -q install -e ".[models]"
!git log -1 --format="CourtVision %h: %s"

# Code imported by an earlier run stays in memory after the update above, and mixing it with the
# new files fails (e.g. "Unknown config key"). Drop it so the next import loads the new code.
for m in [m for m in sys.modules if m == "courtvision" or m.startswith("courtvision.")]:
    del sys.modules[m]
importlib.invalidate_caches()

## Configure

In [ ]:
from courtvision import Config, Pipeline

cfg = Config.load("configs/default.yaml")
cfg.models.detector = f"{DRIVE_ROOT}/models/player_detector.pt"
cfg.models.court_keypoints = f"{DRIVE_ROOT}/models/court_keypoints.pt"
if os.path.exists(f"{DRIVE_ROOT}/models/ball_detector.pt"):
    cfg.models.ball_detector = f"{DRIVE_ROOT}/models/ball_detector.pt"
cfg.teams.team_names = ["Lakers", "Spurs"]   # display names; team 0/1 is decided by jersey colour
cfg.output.dir = f"{DRIVE_ROOT}/output"
cfg.output.cache_dir = f"{DRIVE_ROOT}/stubs"  # detections are cached; re-runs skip the GPU pass

# Player body poses from video: a YOLO pose pass, lifted to 3D skeletons in scene.json.
# Adds GPU time plus ~10 ms per player per frame to lift, so turn it on for clips, not full games.
cfg.pose.enabled = True

# Rigged character for the 3D players (your Mixamo-rigged player.glb). The next cell asks for it
# once and keeps it in Drive.
CHARACTER = f"{DRIVE_ROOT}/models/player.glb"

### Player character

The 3D players use your rigged `player.glb`. The first time, the next cell asks you to upload it and saves it to `MyDrive/courtvision/models/player.glb`. After that, every run uses it automatically. To change the character, replace that file. To use the plain mannequin instead, set `cfg.output.character_glb = None`.

In [ ]:
import shutil
if not os.path.exists(CHARACTER):
    from google.colab import files
    print("Upload player.glb (your Mixamo-rigged character). It will be saved to Drive for later runs.")
    up = files.upload()
    shutil.move(next(n for n in up if n.lower().endswith(".glb")), CHARACTER)
cfg.output.character_glb = CHARACTER
print("3D players will use", CHARACTER, f"({os.path.getsize(CHARACTER) / 1e6:.0f} MB)")

## Pick a video (upload, or put it in `MyDrive/courtvision/input_videos/`)

In [ ]:
import glob
videos = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))
if not videos:
    from google.colab import files
    up = files.upload()
    videos = list(up)
VIDEO = videos[0]
VIDEO

## Run

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
pipe = Pipeline(cfg)
result = pipe.run(VIDEO)
OUT = os.path.join(cfg.output.dir, os.path.splitext(os.path.basename(VIDEO))[0])
print(f"{len(result.players)} frames · {len(result.team_of)} player tracks · "
      f"{len(result.possessions)} touches · {len(result.events)} events")

## Touch-by-touch data (the main output)

One row per touch: a player's continuous control of the ball, from catch to release.

| Columns | What they hold |
|---|---|
| `touch_id`, `possession_id`, `start_frame`/`end_frame`, `start_s`/`end_s`, `duration_s` | when; `possession_id` goes up each time the other team gets the ball |
| `player_id`, `team`, `jersey`, `name` | who (jersey and name stay empty until a roster mapping is set, see below) |
| `start_action` | how the touch began: `catch`, `steal`, `offensive_rebound`, `defensive_rebound`, `recovery`, `start_of_clip` |
| `action`, `outcome`, `target_id`, `target_name` | what they did: `pass` (completed / intercepted, with the receiver), `shot` (made / missed), `turnover`, `loose_ball`, `fumble`, `end_of_clip` |
| `dribbles`, `shot_points`, `shot_distance_m`, `pass_air_s` | detail on the action |
| `start_x_m`, `start_y_m`, `x_m`, `y_m`, `speed_kmh`, `max_speed_kmh`, `distance_covered_m`, `dist_to_basket_m` | the ball handler, in court metres and km/h; `x_m`/`y_m`/`speed_kmh` are at the action frame |
| `nearest_defender_id`, `nearest_defender_m`, `defenders_within_2m`, `players_tracked` | pressure on the ball |
| `tm1_*` … `tm4_*`, `op1_*` … `op5_*` | every other player at the action frame (teammates, then opponents, nearest first): `id`, `x_m`, `y_m`, `speed_kmh`, `dist_m` to the handler |
| `confidence` | 0 to 1: detector confidence, court calibration coverage, touch length, and whether the start and end matched a detected event |

Broadcast cameras rarely show all ten players, so the slots for players off screen are empty.

**Names and jerseys:** set `pipe.player_info = {track_id: {"jersey": "23", "name": "LeBron James"}}` before `pipe.run`, or rebuild the table afterwards with the cell below.

In [ ]:
import pandas as pd
touches = pd.read_parquet(f"{OUT}/touches.parquet")
core = ["touch_id", "possession_id", "start_s", "player_id", "team", "start_action", "action", "outcome",
        "target_id", "dribbles", "x_m", "y_m", "speed_kmh", "nearest_defender_m", "confidence"]
print(f"{len(touches)} touches, {touches.shape[1]} columns")
display(touches[core].head(30))

In [ ]:
# Everyone on the floor at one touch: the ball handler plus each teammate and opponent slot.
t = touches.iloc[0]
slots = [("handler", t.player_id, t.x_m, t.y_m, t.speed_kmh, 0.0)]
for prefix, n in (("tm", 4), ("op", 5)):
    for k in range(1, n + 1):
        if pd.notna(t[f"{prefix}{k}_id"]):
            slots.append((f"{prefix}{k}", *(t[f"{prefix}{k}_{c}"] for c in ("id", "x_m", "y_m", "speed_kmh", "dist_m"))))
print(f"touch {t.touch_id}: player {t.player_id} {t.start_action} -> {t.action} ({t.outcome})")
display(pd.DataFrame(slots, columns=["slot", "player_id", "x_m", "y_m", "speed_kmh", "dist_to_handler_m"]))

In [ ]:
# Optional: add names once you know which track is which player, then rewrite the touches files.
from courtvision.events.touches import touches_table
pipe.player_info = {}   # e.g. {7: {"jersey": "23", "name": "LeBron James"}}
if pipe.player_info:
    touches = touches_table(result.possessions, result, cfg.teams.team_names, pipe.court, pipe.player_info)
    touches.to_parquet(f"{OUT}/touches.parquet", index=False)
    touches.to_csv(f"{OUT}/touches.csv", index=False)

In [ ]:
# Per-player summary of the touches, then the event log and box score.
display(pd.crosstab([touches.player_id, touches.team], touches.action, margins=True))
events = pd.read_csv(f"{OUT}/events.csv")
display(events.head(30))
display(pd.read_csv(f"{OUT}/box_score.csv"))

## Tracking data (metres, every player and the ball, every frame)

In [ ]:
tracking = pd.read_csv(f"{OUT}/tracking.csv")
display(tracking.head())
tracking[tracking.object == "player"].groupby("id")[["distance_m", "speed_kmh"]].max().sort_values("distance_m", ascending=False)

## 3D play reconstruction

Animated players: speed drives the gait, defenders face the ball, shots and passes trigger jumps and throws, and measured skeletons from the pose stage replace the synthesized limbs. Use **Free orbit** to move the camera; **Open scene.json** and **Load character** also work on the saved file outside Colab.

In [ ]:
# Served from the kernel instead of pasted into the notebook: the page embeds the character
# (~20 MB), which would bloat the .ipynb and slow Colab down.
import subprocess, sys
from urllib.parse import quote
from google.colab import output
if "viewer_server" not in globals():
    viewer_server = subprocess.Popen([sys.executable, "-m", "http.server", "8765", "--directory", cfg.output.dir],
                                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import time; time.sleep(1)  # let the server start
output.serve_kernel_port_as_iframe(8765, path=f"/{quote(os.path.basename(OUT))}/play_3d.html", height=760)

Changed the character or team colours? Rebuild only the 3D page from the saved `scene.json`, with no need to rerun the models:

In [ ]:
import json
from courtvision.recon3d.viewer import scene_to_threejs
scene = json.load(open(f"{OUT}/scene.json"))
colors = ["#%02x%02x%02x" % tuple(reversed(c)) for c in pipe.team_colors_bgr]
scene_to_threejs(scene, f"{OUT}/play_3d.html", team_colors=colors, character_glb=cfg.output.character_glb)
output.serve_kernel_port_as_iframe(8765, path=f"/{quote(os.path.basename(OUT))}/play_3d.html", height=760)

In [ ]:
# Save the 3D play to open in any browser (it works offline apart from loading three.js).
from google.colab import files
files.download(f"{OUT}/play_3d.html")

## Annotated video

In [ ]:
# Re-encode to H.264 so the browser can play it inline.
src = f"{OUT}/{os.path.splitext(os.path.basename(VIDEO))[0]}_annotated.mp4"
!ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -crf 26 /content/preview.mp4
from IPython.display import Video
Video("/content/preview.mp4", embed=True, width=960)

## Touch-by-touch preview and download

In [ ]:
display(touches.head(10))
from google.colab import files
files.download(f"{OUT}/touches.csv")